# 08 - Points by Neighborhood

For a target neighborhood, builds a hex grid and counts business openings and closings per hex cell per year. Loads the full openings/closings dataset and SF neighborhood boundaries, spatially joins points to neighborhoods, then bins events into a 100m hex grid clipped to the neighborhood boundary. Exports the hex-aggregated GeoDataFrame as a parquet for use in map visualizations.

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import json
from shapely.geometry import Polygon

In [2]:
gdf_sf = gpd.read_parquet('../../data/processed/ALL_openings_closings.parquet')

if gdf_sf.crs is None:
    gdf_sf = gdf_sf.set_crs(epsg=4326)

gdf_sf.head()

,uniqueid,business_account_number,location_id,ownership_name,dba_name,business_start_date,business_end_date,location_start_date,location_end_date,naics_code,naics_code_description,lic_code,lic_code_description,business_corridor,neighborhoods_analysis_boundaries,geometry,administratively_closed,year,status
931,1239658-12-191-1109070,1109070,1239658-12-191,"Emptor.Ai, Inc.",Openwrench,2019-12-01,2023-06-30,2019-12-01,2023-06-30,5100-5199,Information,None,None,None,West of Twin Peaks,POINT (-122.43779 37.73136),***Administratively Closed,2019,opened
937,1293514-12-211-1130431,1130431,1293514-12-211,Jorge Ellington,Esencia Lje,2021-12-15,2022-05-31,2021-12-15,2022-05-31,None,None,None,None,None,Bernal Heights,POINT (-122.42358 37.73539),None,2021,opened
948,1335196-06-231-1147567,1147567,1335196-06-231,Cara Ballard,Petitesmuses,2023-06-08,NaT,2023-06-08,NaT,4400-4599,Retail Trade,None,None,None,Outer Richmond,POINT (-122.49131 37.77437),None,2023,opened
952,1401440-10-251-1175256,1175256,1401440-10-251,Alejandrina F Alvarado Castro,Alejandrina's Cleaning,2025-10-06,NaT,2025-10-06,NaT,None,None,None,None,None,Bayview Hunters Point,POINT (-122.39642 37.73588),None,2025,opened
956,1255058-07-201-1115489,1115489,1255058-07-201,S2/H Construction,S2/H Construction,2020-06-01,NaT,2020-06-01,NaT,2300-2399,Construction,None,None,None,Bayview Hunters Point,POINT (-122.38897 37.7275),None,2020,opened


In [3]:
sf_neighs = gpd.read_file('../../data/raw/sf_neighborhoods.geojson')

if sf_neighs.crs is None:
    sf_neighs = sf_neighs.set_crs(epsg=4326)

sf_neighs = sf_neighs.to_crs(epsg=4326)

print(sf_neighs.columns)
print(gdf_sf.columns)

Index([':id', ':version', ':created_at', ':updated_at', 'nhood', 'geometry'], dtype='object')
Index(['uniqueid', 'business_account_number', 'location_id', 'ownership_name',
       'dba_name', 'business_start_date', 'business_end_date',
       'location_start_date', 'location_end_date', 'naics_code',
       'naics_code_description', 'lic_code', 'lic_code_description',
       'business_corridor', 'neighborhoods_analysis_boundaries', 'geometry',
       'administratively_closed', 'year', 'status'],
      dtype='object')


In [4]:
sf_neighs = sf_neighs.rename(columns={'nhood': 'neighborhood'})

gdf_sf = gdf_sf.drop(columns=["neighborhood", "nhood", "index_right"], errors="ignore")

gdf_sf_neigh = gpd.sjoin(
    gdf_sf,
    sf_neighs[["neighborhood", "geometry"]],
    how="left",
    predicate="within"
)

print(gdf_sf_neigh["neighborhood"].value_counts().head(10))

neighborhood
Financial District/South Beach    26722
Mission                           14154
South of Market                   11940
Bayview Hunters Point              8647
Sunset/Parkside                    8201
Castro/Upper Market                5688
Marina                             5585
Outer Richmond                     5301
Chinatown                          5208
Tenderloin                         5173
Name: count, dtype: int64


In [5]:
gdf_sf_neigh["location_start_date"] = pd.to_datetime(
    gdf_sf_neigh["location_start_date"], errors="coerce"
)
gdf_sf_neigh["location_end_date"] = pd.to_datetime(
    gdf_sf_neigh["location_end_date"], errors="coerce"
)

open_points = gdf_sf_neigh[
    (gdf_sf_neigh["location_start_date"] >= "2019-01-01") &
    (gdf_sf_neigh["location_start_date"] < "2026-01-01")
].copy()

close_points = gdf_sf_neigh[
    (gdf_sf_neigh["location_end_date"] >= "2019-01-01") &
    (gdf_sf_neigh["location_end_date"] < "2026-01-01")
].copy()

open_points["status"] = "Opening"
close_points["status"] = "Closing"

open_points["event_year"] = open_points["location_start_date"].dt.year
close_points["event_year"] = close_points["location_end_date"].dt.year

In [ ]:
# Change target_neighborhood and output_file for each run
target_neighborhood = "Financial District"
output_file = "../../data/processed/fidi_hex_open_close.parquet"

if target_neighborhood not in gdf_sf_neigh["neighborhood"].dropna().unique():
    target_neighborhood = (
        close_points
        .groupby("neighborhood")
        .size()
        .sort_values(ascending=False)
        .index[0]
    )

print("Mapping neighborhood:", target_neighborhood)

point_events = pd.concat([
    open_points[open_points["neighborhood"] == target_neighborhood],
    close_points[close_points["neighborhood"] == target_neighborhood]
], ignore_index=True)

point_events = point_events[point_events.geometry.notna()].copy()
point_events = gpd.GeoDataFrame(point_events, geometry="geometry", crs=gdf_sf_neigh.crs).to_crs(epsg=4326)

point_events["lon"] = point_events.geometry.x
point_events["lat"] = point_events.geometry.y

top_boundary = sf_neighs[
    sf_neighs["neighborhood"] == target_neighborhood
].to_crs(epsg=4326)

In [7]:
points_proj = point_events.to_crs(epsg=7131)
boundary_proj = top_boundary.to_crs(epsg=7131)

hex_size = 100
minx, miny, maxx, maxy = boundary_proj.total_bounds

hexagons = []
x = minx
col = 0

while x < maxx:
    y = miny
    if col % 2 == 1:
        y += hex_size * np.sqrt(3) / 2
    while y < maxy:
        hexagon = Polygon([
            (x + hex_size * np.cos(angle), y + hex_size * np.sin(angle))
            for angle in np.linspace(0, 2 * np.pi, 7)[:-1]
        ])
        hexagons.append(hexagon)
        y += hex_size * np.sqrt(3)
    x += hex_size * 1.5
    col += 1

hex_grid = gpd.GeoDataFrame(geometry=hexagons, crs=boundary_proj.crs)
hex_grid = gpd.overlay(
    hex_grid,
    boundary_proj[["neighborhood", "geometry"]],
    how="intersection"
)
hex_grid["hex_id"] = range(len(hex_grid))

In [8]:
points_proj = points_proj.drop(columns=["index_right"], errors="ignore")

points_with_hex = gpd.sjoin(
    points_proj,
    hex_grid[["hex_id", "geometry"]],
    how="inner",
    predicate="intersects"
)

hex_counts = (
    points_with_hex
    .groupby(["hex_id", "event_year", "status"])
    .size()
    .reset_index(name="count")
)

hex_counts_wide = (
    hex_counts
    .pivot_table(
        index=["hex_id", "event_year"],
        columns="status",
        values="count",
        fill_value=0
    )
    .reset_index()
)

if "Opening" not in hex_counts_wide.columns:
    hex_counts_wide["Opening"] = 0
if "Closing" not in hex_counts_wide.columns:
    hex_counts_wide["Closing"] = 0

years = list(range(2016, 2026))
hex_years = (
    pd.MultiIndex.from_product(
        [hex_grid["hex_id"], years],
        names=["hex_id", "event_year"]
    )
    .to_frame(index=False)
)

hex_years = hex_years.merge(hex_counts_wide, on=["hex_id", "event_year"], how="left")
hex_years[["Opening", "Closing"]] = hex_years[["Opening", "Closing"]].fillna(0)

hex_plot = hex_grid.merge(hex_years, on="hex_id", how="left")
hex_plot = hex_plot.to_crs(epsg=4326)

max_opening = max(hex_plot["Opening"].max(), 1)
max_closing = max(hex_plot["Closing"].max(), 1)

In [9]:
hex_plot.to_parquet(output_file)
print(f"Exported {len(hex_plot)} rows to {output_file}")

Exported 6190 rows to ../../data/processed/bhp_hex_open_close.parquet
